In [ ]:
from astropy.coordinates import SkyCoord
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np
import requests
import time
import os

FRITZ_API_KEY = os.environ.get('FRITZ_API_KEY')
if FRITZ_API_KEY is None:
    host = None
    headers = None
    raise ValueError("FRITZ_API_KEY is not set")
else:
    host = "https://fritz.science"
    headers = {'Authorization': f'token {FRITZ_API_KEY}'}


### Milliquas

In [ ]:
# Run with cwd = sample-curation, or adjust path
milliquas_path = Path("../Milliquas/milliquas.txt").resolve()

colspecs = [
    (0, 11),    # 1–11   RA
    (12, 23),   # 13–23  DEC
    (25, 50),   # 26–50  Name
    (51, 55),   # 52–55  Type
    (56, 61),   # 57–61  Rmag
    (62, 67),   # 63–67  Bmag
    (68, 71),   # 69–71  Comment
    (72, 73),   # 73     R
    (74, 75),   # 75     B
    (76, 82),   # 77–82  Z
    (83, 89),   # 84–89  Cite
    (90, 96),   # 91–96  Zcite
    (97, 119),  # 98–119 Xname
    (120, 142), # 121–142 Rname
    (143, 165), # 144–165 Lobe1
    (166, 188), # 167–188 Lobe2
]
names = [
    "RA", "DEC", "Name", "Type", "Rmag", "Bmag", "Comment",
    "R", "B", "Z", "Cite", "Zcite", "Xname", "Rname", "Lobe1", "Lobe2",
]

milliquas = pd.read_fwf(
    milliquas_path,
    colspecs=colspecs,
    names=names,
)
for c in ("Name", "Type", "Comment", "R", "B", "Cite", "Zcite", "Xname", "Rname", "Lobe1", "Lobe2"):
    milliquas[c] = milliquas[c].astype("string").str.strip()
for c in ("RA", "DEC", "Rmag", "Bmag", "Z"):
    milliquas[c] = pd.to_numeric(milliquas[c], errors="coerce")

milliquas = milliquas[['RA', "DEC", "Name", "Type", "Z"]]

milliquas.head()

In [ ]:
all_not_saved = pd.read_csv("../not_saved_sources.txt", sep="\t")

no_junk_not_saved = pd.read_csv("../not_saved_sources_no_junk.txt", sep="\t")

all_not_saved['in_junk'] = all_not_saved['ZTFID'].isin(no_junk_not_saved['ZTFID'])
all_not_saved['MQ_match'] = False
all_not_saved['MQ_name'] = None
all_not_saved['MQ_sep'] = None
all_not_saved['MQ_z'] = None
all_not_saved['MQ_name'] = None
all_not_saved['MQ_type'] = None

In [ ]:
milliquas_coords = SkyCoord(milliquas["RA"], milliquas["DEC"], unit="deg")

not_saved_coords = SkyCoord(all_not_saved['ra'], all_not_saved['dec'], unit="deg")

In [ ]:
for i in tqdm(range(len(not_saved_coords))):
    seps = milliquas_coords.separation(not_saved_coords[i])
    min_sep = np.min(seps)
    if min_sep.arcsec < 1.0:
        all_not_saved.loc[i, 'MQ_match'] = True
        all_not_saved.loc[i, 'MQ_name'] = milliquas.loc[np.argmin(seps), 'Name']
        all_not_saved.loc[i, 'MQ_sep'] = min_sep.arcsec
        all_not_saved.loc[i, 'MQ_z'] = milliquas.loc[np.argmin(seps), 'Z']
        all_not_saved.loc[i, 'MQ_type'] = milliquas.loc[np.argmin(seps), 'Type']


In [ ]:
all_not_saved.to_csv("../not_saved_sources_MQ_match.txt", sep="\t")

### Fritz AGN/QSO

In [ ]:
all_not_saved = pd.read_csv("../not_saved_sources_MQ_match.txt", sep="\t")
all_not_saved['Fritz_match'] = False
all_not_saved['Fritz_classifications'] = None

In [ ]:
for idx in tqdm(range(len(all_not_saved))):
    ztfid = all_not_saved.loc[idx, 'ZTFID']
    try:
        good_type, clf_str = is_fritz_AGN_QSO(ztfid)
    except Exception as e:
        good_type, clf_str = False, f"Error: {e}"
    all_not_saved.loc[idx, 'Fritz_match'] = good_type
    all_not_saved.loc[idx, 'Fritz_classifications'] = clf_str
    time.sleep(0.5)



In [ ]:
all_not_saved.to_csv("../not_saved_sources_AGN_QSO.txt", sep="\t")
